In [14]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

processed_data_path = "../data/processed/"

processed_data = pd.read_csv(os.path.join(processed_data_path, "processed_data.csv"))


In [15]:
X = processed_data.drop(columns=["Churn"])
y = processed_data["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

## Import Baseline Model

In [16]:
baseline_metrics = json.load(open("../models/baseline/baseline_random_forest_metrics.json"))
baseline_model = joblib.load("../models/baseline/baseline_random_forest.joblib")
baseline_metrics

{'Model': 'RandomForestClassifier(default)',
 'Accuracy': 0.7984386089425124,
 'Precision': 0.663003663003663}

## Tuning Baseline Model 

### - Using optuna

In [17]:
import optuna
from sklearn.metrics import accuracy_score, precision_score


def objective(trial):
    params = {
        # 1. จำนวนต้นไม้ในป่า
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        # 2. เกณฑ์วัดความบริสุทธิ์
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        # 3. ความลึกสูงสุดของแต่ละต้น
        'max_depth': trial.suggest_int('max_depth', 3, 32),
        # 4. จำนวนข้อมูลขั้นต่ำในการแตกกิ่ง
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 40),
        # 5. จำนวนข้อมูลขั้นต่ำที่ใบ
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 32),
        # 6. จำนวน Feature ที่สุ่มมาใช้ในแต่ละต้น
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        # 7. สัดส่วนข้อมูล bootstrap ต่อต้น
        'max_samples': trial.suggest_float('max_samples', 0.5, 1.0),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1,
    }

    model = RandomForestClassifier(**params)

    # --- 5-Fold Cross-Validation (train set เท่านั้น) ---
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

    return scores.mean()

In [18]:
# สั่งให้ Optuna รันการค้นหา
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-09-07 18:40:41,096] A new study created in memory with name: no-name-ca97d700-ee1e-428f-bd4f-ad2d5dcff202


[I 2026-09-07 18:40:43,274] Trial 0 finished with value: 0.8298467838681928 and parameters: {'n_estimators': 102, 'criterion': 'entropy', 'max_depth': 3, 'min_samples_split': 23, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'max_samples': 0.6727954689437744}. Best is trial 0 with value: 0.8298467838681928.
[I 2026-09-07 18:40:45,492] Trial 1 finished with value: 0.8413743056897488 and parameters: {'n_estimators': 189, 'criterion': 'entropy', 'max_depth': 21, 'min_samples_split': 33, 'min_samples_leaf': 11, 'max_features': 'sqrt', 'max_samples': 0.7293703143143249}. Best is trial 1 with value: 0.8413743056897488.
[I 2026-09-07 18:40:47,390] Trial 2 finished with value: 0.8409467769892929 and parameters: {'n_estimators': 205, 'criterion': 'gini', 'max_depth': 27, 'min_samples_split': 23, 'min_samples_leaf': 12, 'max_features': 'sqrt', 'max_samples': 0.9114485041870484}. Best is trial 1 with value: 0.8413743056897488.
[I 2026-09-07 18:40:47,954] Trial 3 finished with value: 0.8414311050

In [19]:
# แสดงผลลัพธ์
print("="*40)
print(f" Accuracy สูงสุดที่ Optuna หาได้: {study.best_value:.4f}")
print(" ค่า Hyperparameters ที่ดีที่สุดของ Decision Tree:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")
print("="*40)

 Accuracy สูงสุดที่ Optuna หาได้: 0.8422
 ค่า Hyperparameters ที่ดีที่สุดของ Decision Tree:
  - n_estimators: 295
  - criterion: entropy
  - max_depth: 14
  - min_samples_split: 31
  - min_samples_leaf: 14
  - max_features: sqrt
  - max_samples: 0.7118933535103489


In [20]:
# สร้างโมเดลด้วย best_params จาก study
best_model = RandomForestClassifier(**study.best_params, random_state=42, n_jobs=-1)

# เทรนกับ Train Set
best_model.fit(X_train, y_train)

# ประเมินกับ Test Set
y_pred = best_model.predict(X_test)

print(f"Baseline  Accuracy (Test): {baseline_metrics['Accuracy']:.4f}")
print(f"Tuned     Accuracy (Test): {accuracy_score(y_test, y_pred):.4f}")
print(f"Tuned     Precision (Test): {precision_score(y_test, y_pred):.4f}")

Baseline  Accuracy (Test): 0.7984
Tuned     Accuracy (Test): 0.8141
Tuned     Precision (Test): 0.7063


In [21]:
out_dir = "../models/tuned"
os.makedirs(out_dir, exist_ok=True)

joblib.dump(best_model, os.path.join(out_dir, "tuned_random_forest.joblib"))

tuned_metrics = {
    "Model": "RandomForestClassifier(optuna-tuned)",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "best_params": study.best_params,
    "cv_best_accuracy": study.best_value,
    "n_trials": len(study.trials),
}
with open(os.path.join(out_dir, "tuned_random_forest_metrics.json"), "w") as f:
    json.dump(tuned_metrics, f, indent=2)

tuned_metrics

{'Model': 'RandomForestClassifier(optuna-tuned)',
 'Accuracy': 0.8140525195173882,
 'Precision': 0.7063197026022305,
 'best_params': {'n_estimators': 295,
  'criterion': 'entropy',
  'max_depth': 14,
  'min_samples_split': 31,
  'min_samples_leaf': 14,
  'max_features': 'sqrt',
  'max_samples': 0.7118933535103489},
 'cv_best_accuracy': 0.8422211756369883,
 'n_trials': 50}

In [22]:
# ดูจำนวนและสัดส่วนของ Target ในข้อมูล
print("จำนวนข้อมูลแยกตาม Class:")
print(pd.Series(y_test).value_counts())

print("\nสัดส่วน % ของแต่ละ Class:")
print(pd.Series(y_test).value_counts(normalize=True) * 100)

จำนวนข้อมูลแยกตาม Class:
Churn
0    1036
1     373
Name: count, dtype: int64

สัดส่วน % ของแต่ละ Class:
Churn
0    73.527324
1    26.472676
Name: proportion, dtype: float64


In [23]:
from sklearn.metrics import classification_report

# Predict ผลลัพธ์
y_pred = best_model.predict(X_test)

# ดูรายงานสรุป
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.92      0.88      1036
           1       0.71      0.51      0.59       373

    accuracy                           0.81      1409
   macro avg       0.77      0.72      0.74      1409
weighted avg       0.80      0.81      0.80      1409

